In [0]:
import numpy as np 
import matplotlib.pyplot as plt 
import random 

## ### **RANDOM SHAPE GENERATOR**

In [0]:
class DrawShape():
    def __init__(self, n: int) -> None:
        self.n = n

    def generateGrid(self) -> list:
        """
        Generate an empty n x n grid filled with empty values ('.').
        """
        n = self.n 
        axis = np.full(n, 0).tolist()
        output = []
        for _ in range(n):
            output.append(axis.copy())

        return output 


    def placeInitial(self) -> list[list]:
        """
        Place a box in the middle of an empty grid. This will be the cornerstone while building the shape.
        """
        grid = DrawShape.generateGrid(self)
        n = self.n 
        pos = int(np.ceil(n/2)) - 1            #find the index of the middle position
        grid[pos][pos] = 1
        return grid



    def getOpenPositions(self, input_grid) -> list[tuple]:
        n = self.n
        grid = input_grid 


        output = []
        for i in range(n):
            for j in range(n):
                if grid[i][j] == 0:
                    continue 

                if grid[i][j] == 1:
                    try:
                        if grid[i-1][j] == 0:
                            output.append([i-1, j])
                    except IndexError:
                        if grid[n-1][j] == 0:
                            output.append([n-1, j])

                    try:
                        if grid[i+1][j] == 0:
                            output.append([i+1, j])
                    except IndexError:
                        if grid[0][j] == 0:
                            output.append([0, j])

                    try:
                        if (grid[i][j-1] == 0):
                            output.append([i, j-1])
                    except IndexError:
                        if (grid[i][n-1] == 0):
                            output.append([i, n-1])

                    try:
                        if (grid[i][j+1] == 0):
                            output.append([i, j+1])
                    except IndexError:
                        if (grid[i][0] == 0):
                            output.append([i, 0])


        return output
    


    def growShape(self, input_grid) -> list[list]:
        """
        Add a block onto the shape based on random probability.
        """
        grid = input_grid 
        n = self.n
        open_pos = DrawShape.getOpenPositions(self, input_grid = grid )

        #compute a random index 
        a = 0
        b = len(open_pos)
        random_index = random.randint(a, b - 1)

        #find next point 
        next_point = open_pos[random_index]

        #populate grid at next point 
        grid[next_point[0]][next_point[1]] = 1

        return grid 

    
    def createShape(self) -> list[list]:
        n = self.n 
        i = 0 

        grid = DrawShape.placeInitial(self)

        while i < n - 1:
            next_grid = DrawShape.growShape(self, input_grid = grid)
            grid = next_grid 
            i += 1

        return grid 






x = DrawShape(n=5).createShape()
for row in x:
    print(row)
# # print(x)
# print(np.sum(x))
# plt.imshow(x)

# plt.show()

## ### GENERATE BINARY VECTOR AND LATTICE


In [0]:
def generate_input_binary_vector(size: int):
  output = []
  for _ in range(size):
    a = np.random.randint(0, 2, size=1)
    output.append(int(((-1)**a + 1)/2))  #a 1 or -1 is generated - add 1 to bring to 0 or 2 - divide by 2

  return output 



input_vector = generate_input_binary_vector(12)
input_lattice = DrawShape(n=len(input_vector)).createShape()

## ### BINARY VECTOR ENCODER

In [0]:
class BinaryEncoder:
  def __init__(self, input_vector: list[int] = None, input_vec_id: int = None, input_grid: list[list[int]] = None, input_grid_id: str = None) -> None:
    self.vector = input_vector 
    self.vec_id = input_vec_id
    self.grid = input_grid 
    self.grid_id = input_grid_id


  def vectorToDec(self) -> int:
    #flip the input vector so we go the right way when counting 
    vector = self.vector[::-1] 

    #start counting
    decimal_val = 0

    for k in range(len(vector)):
      if vector[k] == 1:
        decimal_val += 2**k
      else: #treat it as 0 otherwise 
        decimal_val += 0

    return decimal_val
  


  def decToVector(self) -> list[int]:
    """
    NOTE THAT THIS FUNCTION DOES NOT PRESERVE SIZE IF PLACING DECIMALS IN RECURSIVELY
    """
    value = self.vec_id 

    vec = []

    while True:
      if int(value%2) == 0:   #if there is no remainder
        vec.append(0)
      if int(value%2) == 1:   #if there is a remainder
        vec.append(1)

      value = np.floor(value/2)   #set new value for quotient 
      if value == 0:
        break 

    return vec[::-1]



  def gridToID(self) -> str:
    grid = self.grid 
    area = np.sum(grid)

    output_id_string = f'gr_s{area}'

    for k in range(len(grid)):
      row = grid[k]
      row_binary_id = BinaryEncoder(input_vector = row).vectorToDec()
      output_id_string += f'_{row_binary_id}r{k}'

    print(output_id_string)
    return output_id_string
  


  def IDToGrid(self) -> list[list[int]]:
    id_string = self.grid_id

    id_str_split = id_string.split("_")

    size = int(id_str_split[1].replace('s', ''))
    id_type = id_str_split[0]
    if id_type != 'gr':
      raise ValueError('Input grid ID is invalid, please ensure type is "gr".')


    fill_vector = np.zeros(size)

    output_grid = []
    for k in range(len(id_str_split[2:])):
      row_code = id_str_split[2:][k]
      binary_decimal, row_num = (row_code.split('r'))     #these are all strings still, ok?
      vec = BinaryEncoder(input_vec_id = int(binary_decimal)).decToVector()

      if k != int(row_num):
        raise ValueError('Error reading row numbers off of input grid ID, please ensure row numbers and vector binary IDs are one-to-one.')


      if len(vec) < size:   #if length is not preserved (it's probably not)
        fill_vec_copy = fill_vector.copy()
        fill_vec_copy[:len(vec)] += vec[::-1]   #add the binary vector to the filled vector of zeros
                                                #note that we need to flip the vector first and then flip it back to preserve the order of the binary vector with the fill array of 0s
                                                
      if len(vec) == size:  #if length is preserved
        fill_vec_copy = fill_vector.copy()
        fill_vec_copy += vec[::-1]
                                
      output_grid.append(fill_vec_copy[::-1])
    
    return list(output_grid)



In [0]:

y = BinaryEncoder(input_grid = input_lattice)
gid = y.gridToID()
z = BinaryEncoder(input_grid_id=gid)
lat = z.IDToGrid()



plt.imshow(lat)
plt.show()
plt.imshow(input_lattice)
plt.show()

## ### **ISING MODEL ALGORITHM**

In [0]:
class IsingEvolution():
    def __init__(self, vector: list[int], lattice: list[list[int]]) -> None:
        self.vector = vector 
        self.lattice = lattice 

    def mapper(self) -> list[list[int]]:
        grid = np.copy(self.lattice) 
        vector = np.copy(self.vector) 

        random.shuffle(vector)


        dim = len(vector) 
        if dim != np.sum(grid):
            print(dim, np.sum(grid))
            raise ValueError(f'Number of input entries and mapping entries are inconsistent.')

        n, m = np.shape(grid)
        while (dim - 1) > -1:
            for i in range(n):
                for j in range(m):
                    if grid[i][j] == 1:
                        grid[i][j] = ((vector[dim-1] * 2) - 1) #bring 0 or 1 to 0 or 2 -- subtract 1 to bring to -1 or 1
                        dim -= 1 
                        continue

        return grid 


    def hamiltonian(self, lattice_configuration: list[list[int]], J: float = 1) -> float:

        n, m = np.shape(lattice_configuration)
        E = 0 #set base energy 
        passed_sites = []
        
        #search the lattice configuration
        for i in range(n):
            for j in range(m):
                site = lattice_configuration[i][j]

                if site == 0:       #site is not within lattice 
                    continue 

                else:               #site is within lattice and is spin-up or spin-down
                    try:  
                        E_down = lattice_configuration[i][j] * lattice_configuration[i][j+1] 
                    except IndexError:
                        #for edge conditions; e.g. if the site is at an edge, just loop it over to other side (repeatable grid)
                        E_down = lattice_configuration[i][j] * lattice_configuration[i][0]

                    try:
                        E_right = lattice_configuration[i][j] * lattice_configuration[i+1][j] 
                    except IndexError:
                        #for edge conditions; e.g. if the site is at an edge, just loop it over to other side (repeatable grid)
                        E_right = lattice_configuration[i][j] * lattice_configuration[0][j] 


                    
                E = E + E_down + E_right
                
        return -E*J
    



    def isingEvolve(self, input_configuration: list[list[int]]) -> None:
        #determine all active sites with an atom 
        n, m = np.shape(input_configuration)
        active_sites = []

        for i in range(n):
            for j in range(m):
                if input_configuration[i][j] != 0:
                    active_sites.append((i, j))

        #copy the base configuration
        configuration = np.copy(input_configuration)

        iterations = 0

        #algorithm
        while True:    
            #1 - choose a random site ; compute energy
            rand_site_idx = random.randint(0, len(active_sites)-1)
            a = active_sites[rand_site_idx][0]
            b = active_sites[rand_site_idx][1]

            E1 = IsingEvolution.hamiltonian(self, lattice_configuration = configuration)

            #2 - flip the random spin site ; compute energy
            next_configuration = np.copy(configuration)
            next_configuration[a][b] = (-1)*(next_configuration[a][b])

            E2 = IsingEvolution.hamiltonian(self, lattice_configuration = next_configuration)

            #compare the energies 
            if E2 > E1: 
                continue
        
            else:
                configuration = np.copy(next_configuration)

        

            iterations += 1

            plt.imshow(configuration)
            plt.show()

            #check if the configuration has matured to convergence
            if np.abs(np.sum(configuration)) == np.sum(self.lattice):
                print(f'convergence achieved: net spin {np.sign(np.sum(configuration))} ({iterations} iterations).' )
                break

        



        
                
                    
x = IsingEvolution(vector = input_vector, lattice = input_lattice)
config = x.mapper()
x.isingEvolve(input_configuration = config)



